# Conditional models: poses given keypoints, with occlusions


How the keypoint conditions work: sample poses from p(pose | visible keypoints) of a conditional model, for a test pose and a group of hidden joints, and look at them as in Fig. 3 of the HuProSO3 paper (the samples on top of each other, the less likely ones more transparent).

**Models.** Every conditional model on 3D keypoints that is there:
- the authors' flows of the paper's code (`experiments/pretrained_models`): `id_0384`, trained with random masks of hidden joints (§4.3, one model for every occlusion), and `id_0410`, trained with all joints visible (§4.2; hidden joints are outside of what it has seen);
- our conditional runs of the model zoo (e.g. RSGM, `explore/training/rsgm_cond_train.ipynb`): every run below `EXPERIMENTS` with `condition = "3D"`, at its best checkpoint.

**Sections.** 1. the models · 2. the interactive panel: `N_COMPARE` models side by side, occlusion (a preset of the paper or any joints), test pose, seed, and a table that compares them · 3. the occlusions of the paper on many test poses for every model, next to Tab. 3.

**Conventions** (those of the paper's code):
- the condition is the 66 numbers of the forward kinematics (22 joints × 3, meters, root at the origin); a hidden joint is a keypoint set to 0 for the flows (they have no mask input) and NaN for our models (their mask input);
- MPJPE is the mean over the 22 keypoints (the root included), MGEO the mean geodesic angle over the 19 joints;
- "mean of 10" is one pose: the mean rotation of every joint over the 10 samples (the chordal mean, as `scipy Rotation.mean`).

**Reproducibility.** A picture is fixed by the model, the hidden joints, the index of the test pose and the seed; the caption of every picture says them.

In [ ]:
import json
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import roma
import torch
from IPython.display import display
from omegaconf import OmegaConf

import hp.estimators.rotnorm.utils.sd as sd
from hp.data.amass_cfg import AmassJoints, AmassJoints4Learning
from hp.data.conditions import keypoints
from hp.estimators.rotnorm.flow.flow_nd import get_nd_flow
from hp.estimators.simple_mlp import MLPTransform
from hp.evaluation.data import load_reference
from hp.models.io import MANIFEST, load_model
from hp.training import best_checkpoint
from hp.utils.paths import REPO_ROOT, get_path
from hp.utils.skeleton_forward_kinematics import perform_forward_kinematics

# ---- parameters ----------------------------------------------------------------------
PRETRAINED = REPO_ROOT / "experiments" / "pretrained_models"   # the authors' models (checkpoints of the paper's code)
EXPERIMENTS = "~/data/experiments"          # searched for our conditional runs (model.json with condition "3D")
TEST_ROTATIONS = "~/data/AMASS_rotations"   # the official test split (Transitions + SSM), all frames
# The occlusions of the paper (Tab. 3; the joint names of the paper's code, explore/evaluate.ipynb), by the keypoints
# they hide. Add your own: any names of AmassJoints (the first 22).
OCCLUSIONS = {
    "none": [],
    "L (left leg)": ["LeftUpLeg", "LeftLeg", "LeftFoot"],
    "A+H (left arm and hand)": ["LeftArm", "LeftElbow", "LeftHand"],
    "S+UA (right shoulder, upper arm)": ["RightShoulder", "RightArm"],
}
DEVICES = {"flow": "cpu", "rsgm": "mps", "poserfm": "mps", "posedflow": "mps"}     # by model type
VIEWS = {"front and 45°": (0, 45), "front and side": (0, 90), "front": (0,), "side": (90,)}   # azimuths of the renders
SAVE_DIR = "~/data/conditional_samples"     # pictures saved from the panel
N_COMPARE = 2                              # models side by side in the panel
DEFAULT_MODELS = None                       # their first choice (names of section 1); None: the authors' id_0384 and ours
# section 3
EVAL_MODELS = None                          # names of the models to evaluate (see section 1); None: all
N_EVAL = 1000                               # test poses
K = 10                                      # samples per pose ("mean of K", "best of K")
SEED = 0
# ----------------------------------------------------------------------------------------

expand = lambda p: Path(p).expanduser()
J = len(AmassJoints4Learning)
NAMES22 = [AmassJoints(i).name for i in range(22)]

## 1. The models

Every model is wrapped to the same three calls, so that the panel and the evaluation do not care what it is:
- `sample(kp, hidden, k, seed)`: k poses [B, k, 19, 3, 3] for each keypoint condition kp [B, 22, 3] with the joints `hidden` (indices 0-21) not observed, and their log-likelihoods [B, k] (`None` if `with_log_prob=False`);
- `log_prob(poses, kp, hidden)`: log p(pose | visible keypoints) [B];
- `masks`: whether it was trained with hidden joints.

The flows are built from the config of their run and loaded from their two checkpoints, as `CondSO3PriorTrainer.load_model` does (without the trainer, which would load the datasets). Their inverse pass returns −log p (checked against `log_prob` of the samples), so the sign is turned. Our models get the hidden joints as NaN keypoints and compute log p with the probability flow ODE (slower: ~30 ms per pose on mps).

In [ ]:
def seeded(seed):
    # a context in which torch draws from `seed`; the random state outside is left as it was
    devices = [torch.device("cuda")] if torch.cuda.is_available() else []
    ctx = torch.random.fork_rng(devices=devices)
    ctx.__enter__()
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    return ctx


class PaperFlow:
    """A conditional flow of the paper's code (CondSO3PriorTrainer): the flow and its condition MLP."""
    kind = "flow"

    def __init__(self, folder):
        folder = Path(folder)
        config = OmegaConf.load(folder / "config.yaml")
        so3 = config.so3
        so3.n_dim, so3.save_path_dir = J, str(folder)
        self.device = DEVICES["flow"]
        self.flow = get_nd_flow(so3).to(self.device)
        self.flow.load_state_dict(torch.load(folder / "model_ckpt_pdf_estimator.pt", map_location=self.device))
        self.flow.permute_dims = np.loadtxt(folder / "permute_dims.pt")
        self.mlp = MLPTransform(so3.dim_cond_feature, so3.feature_channels, Nh=config.conditioning.hidden_mlp_dims,
                                num_hid_layers=config.conditioning.num_hid_layers).to(self.device)
        self.mlp.load_state_dict(torch.load(folder / "model_ckpt_feature_extractor.pt", map_location=self.device))
        self.flow.eval(), self.mlp.eval()
        self.masks = bool(config.conditioning.mask)
        self.negative_offset = bool(config.conditioning.get("mask_neg_offset", False)) and self.masks
        self.description = (f"the authors' flow {folder.name}, "
                            + ("trained with random masks of hidden joints" if self.masks else "trained with all joints visible"))

    def features(self, kp, hidden):
        kp = kp.clone()
        kp[:, list(hidden)] = -1e5 if self.negative_offset else 0.0
        return self.mlp(kp.reshape(len(kp), -1).to(self.device))

    @torch.no_grad()
    def sample(self, kp, hidden=(), k=1, seed=0, with_log_prob=True):
        f = self.features(kp, hidden).repeat_interleave(k, 0)
        ctx = seeded(seed)
        try:
            base = torch.stack([sd.generate_queries(len(f), mode="random") for _ in range(J)]).permute(1, 0, 2, 3).to(self.device)
        finally:
            ctx.__exit__(None, None, None)
        poses, neg_log_p = self.flow.inverse(base, f)  # the inverse pass returns -log p
        return poses.reshape(len(kp), k, J, 3, 3).cpu(), (-neg_log_p.reshape(len(kp), k).cpu() if with_log_prob else None)

    @torch.no_grad()
    def log_prob(self, poses, kp, hidden=()):
        return self.flow(poses.to(self.device), feature=self.features(kp, hidden))[1].cpu()


class ZooModel:
    """A conditional model of the model zoo (condition "3D"): hidden joints are NaN keypoints."""

    def __init__(self, folder, weights="weights_ema"):
        folder = Path(folder)
        self.checkpoint = best_checkpoint(folder) or folder
        if not (self.checkpoint / f"{weights}.pt").is_file():
            weights = "weights"
        kind = json.loads((self.checkpoint / MANIFEST).read_text())["build_args"]["name"]
        self.kind, self.device = kind, DEVICES.get(kind, "cpu")
        self.model = load_model(self.checkpoint, device=self.device, weights=weights)
        if hasattr(self.model, "likelihood_steps"):
            self.model.likelihood_steps = 200
        self.masks = getattr(self.model, "cond_mask", "none") != "none"
        self.description = (f"{kind} {self.checkpoint.relative_to(expand(EXPERIMENTS))} [{weights}], "
                            + ("trained with random masks of hidden joints" if self.masks else "trained with all joints visible"))

    @staticmethod
    def condition(kp, hidden):
        kp = kp.clone()
        kp[:, list(hidden)] = float("nan")
        return kp.reshape(len(kp), -1)

    @torch.no_grad()
    def sample(self, kp, hidden=(), k=1, seed=0, with_log_prob=True):
        cond = self.condition(kp, hidden).repeat_interleave(k, 0)
        ctx = seeded(seed)
        try:
            poses = torch.cat([self.model.sample(len(c), c) for c in cond.split(5000)]).cpu()
        finally:
            ctx.__exit__(None, None, None)
        log_p = None
        if with_log_prob:
            log_p = torch.cat([self.model.log_prob(p.to(self.device), c) for p, c in zip(poses.split(500), cond.split(500))]).cpu()
            log_p = log_p.reshape(len(kp), k)
        return poses.reshape(len(kp), k, J, 3, 3), log_p

    @torch.no_grad()
    def log_prob(self, poses, kp, hidden=()):
        cond = self.condition(kp, hidden)
        return torch.cat([self.model.log_prob(p.to(self.device), c) for p, c in zip(poses.split(500), cond.split(500))]).cpu()


def find_models():
    # name -> a function that loads the model (loaded on first use and kept)
    found = {}
    for config_file in sorted(PRETRAINED.glob("*/config.yaml")):
        c = OmegaConf.load(config_file)
        if c.so3.get("condition") and c.conditioning.get("conditioning_modality") == "3D":
            found[f"paper flow {config_file.parent.name}"] = lambda f=config_file.parent: PaperFlow(f)
    root = expand(EXPERIMENTS)
    for manifest in sorted(root.rglob(MANIFEST)):
        folder = manifest.parent
        if folder.name.startswith("checkpoint_"):
            continue
        args = json.loads(manifest.read_text())["build_args"]
        if args.get("condition") != "3D":
            continue
        state = "" if (folder / "history.json").is_file() else " (unfinished)"
        found[f"{args['name']} {folder.relative_to(root)}{state}"] = lambda f=folder: ZooModel(f)
    return found


LOADERS = find_models()
loaded = {}


def get_model(name):
    if name not in loaded:
        loaded[name] = LOADERS[name]()
    return loaded[name]


test = load_reference("test", TEST_ROTATIONS, "original")
difference = (keypoints(test[:500]) - perform_forward_kinematics(test[:500], selected_joints=AmassJoints4Learning)).abs().max()
print(f"test: {len(test):,} poses; our keypoints vs the paper's FK on 500 poses: max difference {difference * 1000:.4f} mm\n")
print(f"conditional models on 3D keypoints (the authors' in {PRETRAINED}, ours below {expand(EXPERIMENTS)}):")
for name in LOADERS:
    print(f"  {name}")

## 2. Interactive panel: compare models




`N_COMPARE` models (default 2) sample the same test pose with the same hidden joints and the same seed, and are shown side by side.

- **model 1 … N**: one of section 1 each (loaded on first use); **refresh models** looks for them again, e.g. for a run that finished after section 1 was run. Errors are printed below the panel.
- **occlusion**: a preset of the paper, which fills **hidden** (the keypoints that are not observed; Ctrl/Cmd-click to change it).
- **pose**: the index of the test pose (0 … 117,678); **random pose** draws a new index.
- **seed**: the seed of the samplers; **samples**: how many poses each model draws.
- **views**: the azimuths of the renders; **opacity by log p**: the less likely samples of a model more transparent (as the paper; scaled within each model), else all the same.

**Sample** shows:
1. the ground truth (hidden joints as red dots) and, for every model, its samples on top of each other (meshes if the SMPL-X body model is in `body_models`, skeletons otherwise);
2. a table that compares the models: MPJPE of the visible, the hidden and all keypoints and MGEO (mean over the samples), the same for the mean pose of the samples and for the best sample, the spread of the hidden keypoints, log p of the ground truth, and the sampling time;
3. optionally the error per joint of every model (**error per joint**) and the table of every sample (**every sample**).

**Save** writes the picture as a PNG to `SAVE_DIR`; nothing is overwritten.

In [ ]:
BODY_MODEL = Path(get_path("body_models")) / "smplx" / "SMPLX_NEUTRAL.npz"
body_model = None
if BODY_MODEL.is_file():
    from hp.visualization.pv_render import render_overlay
    from hp.visualization.render_tools import load_body_model
    body_model = load_body_model()
    FACES = body_model.f.detach().cpu().numpy()
else:
    print(f"no body model at {BODY_MODEL}: skeletons instead of meshes")
PARENTS = [-1, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 9, 9, 12, 13, 14, 16, 17, 18, 19]


def mean_rotation(poses):
    # the chordal mean (as scipy Rotation.mean) over dim 1 of poses [B, k, J, 3, 3] -> [B, J, 3, 3]
    q = roma.rotmat_to_unitquat(poses.double())
    top = torch.linalg.eigh(torch.einsum("bkji,bkjl->bjil", q, q)).eigenvectors[..., -1]
    return roma.unitquat_to_rotmat(top).float()


def vertices(poses):
    # mesh vertices [N, V, 3] and the 22 body joints [N, 22, 3] of the mesh, of poses [N, 19, 3, 3]: SMPL-X with the
    # 21 body joints, the toes (10, 11) at rest
    pose_body = torch.zeros(len(poses), 21, 3)
    pose_body[:, [j.value - 1 for j in AmassJoints4Learning]] = roma.rotmat_to_rotvec(poses.float().cpu())
    with torch.no_grad():
        body = body_model(pose_body=pose_body.reshape(len(poses), 63))
    return body.v.cpu().numpy(), body.Jtr[:, :22].cpu().numpy()


def crop(images, margin=0.04):
    # the same crop of the white background for all images [..., h, w, 3], so that they keep their scale
    used = (images.reshape(-1, *images.shape[-3:]) < 250).any(-1).any(0)
    rows, cols = np.where(used.any(1))[0], np.where(used.any(0))[0]
    if len(rows) == 0:
        return images
    h, w = used.shape
    r0, r1 = max(rows[0] - int(margin * h), 0), min(rows[-1] + int(margin * h) + 1, h)
    c0, c1 = max(cols[0] - int(margin * w), 0), min(cols[-1] + int(margin * w) + 1, w)
    return images[..., r0:r1, c0:c1, :]


def opacities(log_p, by_likelihood):
    if not by_likelihood or log_p is None or len(log_p) < 2 or float(log_p.max() - log_p.min()) == 0:
        return np.full(len(log_p) if log_p is not None else 1, 0.35)
    x = ((log_p - log_p.min()) / (log_p.max() - log_p.min())).numpy()
    return 0.12 + 0.6 * x                                   # the most likely sample 0.72, the least likely 0.12


def draw_skeletons(ax, kps, azimuth, alphas, color, lw):
    a = np.deg2rad(azimuth)
    x = kps[..., 0] * np.cos(a) + kps[..., 2] * np.sin(a)    # the horizontal axis of a camera turned by `azimuth`
    for k in range(len(kps)):
        for j in range(1, 22):
            p = PARENTS[j]
            ax.plot([x[k, j], x[k, p]], [kps[k, j, 1], kps[k, p, 1]], color=color, lw=lw, alpha=float(alphas[k]), solid_capstyle="round")
    ax.set_aspect("equal")
    ax.axis("off")


def picture(gt, results, hidden, azimuths, by_likelihood, title, caption):
    # the ground truth and, for every result (label, samples [N, 19, 3, 3], log_p [N]), its samples on top of each
    # other: one column each, one row per view, all with the same framing
    n_views, columns = len(azimuths), 1 + len(results)
    alphas = [opacities(r["log_p"], by_likelihood) for r in results]
    aspect = 1.4                                            # height / width of a picture (skeletons)
    if body_model is not None:
        v_gt, joints_gt = vertices(gt)
        v_models = [vertices(r["samples"])[0] for r in results]
        allv = np.concatenate([v_gt, *v_models]).reshape(-1, 3)
        lo, hi = allv.min(0), allv.max(0)
        frame = dict(center=(lo + hi) / 2, radius=np.linalg.norm(hi - lo) / 2, azimuths_deg=azimuths)
        marked = joints_gt[0, list(hidden)]                 # the hidden joints, as red translucent dots on top
        images = crop(np.stack([render_overlay(v_gt, FACES, points=marked, **frame)]
                               + [render_overlay(v, FACES, a, **frame) for v, a in zip(v_models, alphas)]))
        aspect = images.shape[-3] / images.shape[-2]
    fig, axes = plt.subplots(n_views, columns, figsize=(2.6 * columns, 2.6 * aspect * n_views + 0.8), squeeze=False)
    if body_model is not None:
        for r in range(n_views):
            for c in range(columns):
                axes[r, c].imshow(images[c, r])
                axes[r, c].axis("off")
    else:
        kp_gt = keypoints(gt).numpy()
        for r, az in enumerate(azimuths):
            draw_skeletons(axes[r, 0], kp_gt, az, [1.0], "k", 2.4)
            a = np.deg2rad(az)
            axes[r, 0].scatter(kp_gt[0, hidden, 0] * np.cos(a) + kp_gt[0, hidden, 2] * np.sin(a), kp_gt[0, hidden, 1],
                               s=160, color="red", alpha=0.45, zorder=5)
            for c, (res, al) in enumerate(zip(results, alphas), start=1):
                draw_skeletons(axes[r, c], keypoints(res["samples"]).numpy(), az, np.clip(al * 1.2, 0.1, 1), f"C{c - 1}", 1.4)
                draw_skeletons(axes[r, c], kp_gt, az, [0.25], "k", 1.0)
    axes[0, 0].set_title("ground truth" + ("\n(red: hidden joints)" if hidden else ""), fontsize=9)
    for c, res in enumerate(results, start=1):
        axes[0, c].set_title(f"{res['label']}\n{len(res['samples'])} samples", fontsize=9)
    fig.suptitle(title, fontsize=9)
    fig.text(0.01, 0.005, caption, fontsize=7, color="0.35", ha="left", va="bottom")
    fig.tight_layout(rect=(0, 0.025, 1, 1))
    return fig


def per_joint(gt, results, hidden):
    # the error per keypoint and per joint rotation of every model: grouped bars, mean ± std over its samples
    fig, axes = plt.subplots(2, 1, figsize=(13, 6.4))
    width = 0.8 / len(results)
    names19 = [j.name for j in AmassJoints4Learning]
    for m, res in enumerate(results):
        err = (keypoints(res["samples"]) - keypoints(gt)).norm(dim=-1) * 1000      # [N, 22] mm
        rot = torch.rad2deg(roma.rotmat_geodesic_distance(res["samples"], gt))      # [N, 19] deg
        shift = (m - (len(results) - 1) / 2) * width
        axes[0].bar(np.arange(22) + shift, err.mean(0), width, yerr=err.std(0), color=f"C{m}", capsize=1.5, label=res["label"])
        axes[1].bar(np.arange(J) + shift, rot.mean(0), width, yerr=rot.std(0), color=f"C{m}", capsize=1.5, label=res["label"])
    axes[0].set_xticks(range(22), NAMES22, rotation=70, ha="right", fontsize=7)
    axes[1].set_xticks(range(J), names19, rotation=70, ha="right", fontsize=7)
    for tick, j in zip(axes[0].get_xticklabels(), range(22)):
        tick.set_color("red" if j in hidden else "k")
    for tick, joint in zip(axes[1].get_xticklabels(), AmassJoints4Learning):
        tick.set_color("red" if joint.value in hidden else "k")
    axes[0].set_ylabel("keypoint error [mm]")
    axes[1].set_ylabel("rotation error [deg]")
    for ax in axes:
        ax.grid(axis="y", alpha=0.3)
        ax.legend(fontsize=8)
    fig.suptitle("error per joint: mean ± std over the samples (red labels: hidden keypoints / their joints)", fontsize=9)
    fig.tight_layout()
    return fig


def summary(gt, res, hidden):
    # the numbers of one model for the comparison table
    kp = keypoints(gt)
    visible = [j for j in range(22) if j not in hidden]
    s = res["samples"]
    kp_s = keypoints(s)
    err = (kp_s - kp).norm(dim=-1) * 1000                                       # [N, 22] mm
    mgeo = roma.rotmat_geodesic_distance(s, gt).mean(-1)                        # [N] rad
    m = mean_rotation(s[None])
    m_err = (keypoints(m) - kp).norm(dim=-1)[0] * 1000
    spread = (kp_s[:, hidden] - kp_s[:, hidden].mean(0)).norm(dim=-1).mean() * 1000 if hidden else None
    nan = float("nan")
    return {
        "MPJPE visible [mm]": err[:, visible].mean(),
        "MPJPE hidden [mm]": err[:, hidden].mean() if hidden else nan,
        "MPJPE all [mm]": err.mean(),
        "MGEO [rad]": mgeo.mean(),
        "mean pose: MPJPE [mm]": m_err.mean(),
        "mean pose: MGEO [rad]": roma.rotmat_geodesic_distance(m, gt).mean(),
        "best sample: MPJPE [mm]": err.mean(-1).min(),
        "best sample: MGEO [rad]": mgeo.min(),
        "spread of hidden keypoints [mm]": spread if spread is not None else nan,
        "log p(ground truth)": res["gt_log_p"],
        "sampling [s]": res["seconds"],
    }

In [ ]:
import time

L = lambda w: widgets.Layout(width=w)
names = list(LOADERS)


def default_models(names, n):
    # the authors' occlusion flow and our models first, then the rest
    preferred = [x for x in names if x == "paper flow id_0384"] + [x for x in names if not x.startswith("paper flow")]
    order = preferred + [x for x in names if x not in preferred]
    return [order[i % len(order)] for i in range(n)] if order else [None] * n


# radio and toggle buttons instead of drop-downs: the drop-downs of ipywidgets cannot be picked from in VS Code
model_boxes = [widgets.RadioButtons(options=names, value=v, description=f"model {i + 1}", layout=L("440px"))
               for i, v in enumerate(DEFAULT_MODELS or default_models(names, N_COMPARE))]
preset_box = widgets.RadioButtons(options=list(OCCLUSIONS), value="A+H (left arm and hand)", description="occlusion", layout=L("360px"))
hidden_box = widgets.SelectMultiple(options=NAMES22, rows=8, description="hidden", layout=L("280px"))
pose_box = widgets.BoundedIntText(value=0, min=0, max=len(test) - 1, description="pose", layout=L("200px"))
random_button = widgets.Button(description="random pose", icon="random")
refresh_button = widgets.Button(description="refresh models", icon="refresh", tooltip="look for models again (e.g. a run that finished meanwhile)")
seed_box = widgets.IntText(value=SEED, description="seed", layout=L("160px"))
n_box = widgets.BoundedIntText(value=10, min=1, max=64, description="samples", layout=L("160px"))
views_box = widgets.ToggleButtons(options=list(VIEWS), description="views", style={"button_width": "110px"})
opacity_box = widgets.Checkbox(value=True, description="opacity by log p")
joints_box = widgets.Checkbox(value=True, description="error per joint")
details_box = widgets.Checkbox(value=False, description="every sample")
sample_button = widgets.Button(description="Sample", button_style="primary", icon="play")
save_button = widgets.Button(description="Save", icon="save")
out, msg = widgets.Output(), widgets.Output()
last = {}
pick = np.random.default_rng()


def set_preset(change=None):
    hidden_box.value = tuple(OCCLUSIONS[preset_box.value])


def random_pose(_):
    pose_box.value = int(pick.integers(len(test)))


def say(text):
    with msg:
        msg.clear_output(wait=True)
        print(text)


def refresh(_):
    global LOADERS
    LOADERS = find_models()
    for box in model_boxes:
        current = box.value
        box.options = list(LOADERS)
        box.value = current if current in LOADERS else (list(LOADERS)[0] if LOADERS else None)
    say(f"{len(LOADERS)} models: " + ", ".join(LOADERS))


def guarded(callback):
    # show the error of a callback in the panel: VS Code and some Jupyter versions swallow them silently
    def wrapper(change):
        try:
            callback(change)
        except Exception:
            import traceback
            with out:
                traceback.print_exc()
            say("error: see below")
    return wrapper


def sample_all(chosen, gt, hidden, n, seed):
    # every chosen model on the same pose, hidden joints and seed
    kp = keypoints(gt)
    results = []
    for k, name in enumerate(chosen):
        say(f"sampling {name} ({k + 1} of {len(chosen)}) ...")
        model = get_model(name)
        start = time.perf_counter()
        samples, log_p = model.sample(kp, hidden, n, seed)
        seconds = time.perf_counter() - start
        results.append(dict(name=name, label=f"{k + 1}: {name}", model=model, samples=samples[0], log_p=log_p[0],
                            gt_log_p=float(model.log_prob(gt, kp, hidden)[0]), seconds=seconds))
    return results


def print_comparison(gt, results, hidden):
    rows = [summary(gt, r, hidden) for r in results]
    head = f"{'':34s}" + "".join(f"{f'model {k + 1}':>14s}" for k in range(len(results)))
    print(head)
    for key in rows[0]:
        values = [float(r[key]) for r in rows]
        best = None
        if not key.startswith(("log p", "sampling", "spread")) and all(np.isfinite(values)) and len(values) > 1:
            best = int(np.argmin(values))                   # lower is better for the errors
        cells = "".join(f"{v:13.3f}{'*' if k == best else ' '}" if "rad" in key else f"{v:13.1f}{'*' if k == best else ' '}"
                        for k, v in enumerate(values))
        print(f"{key:34s}{cells}")
    print("(* the lowest error; MPJPE and MGEO are means over the samples; the spread of the hidden keypoints is their mean "
          "distance to their mean over the samples; log p is not comparable between models of different smoothing)")


def print_samples(gt, res, hidden):
    kp = keypoints(gt)
    visible = [j for j in range(22) if j not in hidden]
    err = (keypoints(res["samples"]) - kp).norm(dim=-1) * 1000
    mgeo = roma.rotmat_geodesic_distance(res["samples"], gt).mean(-1)
    print(f"\n{res['label']}: every sample, most likely first")
    print(f"{'sample':>8s} {'MPJPE visible':>14s} {'MPJPE hidden':>13s} {'MPJPE all':>10s} {'MGEO':>7s} {'log p':>8s}")
    h = lambda e: f"{e[hidden].mean():10.1f} mm" if hidden else f"{'-':>13s}"
    for i in np.argsort(-res["log_p"].numpy()):
        print(f"{i:8d} {err[i, visible].mean():11.1f} mm {h(err[i])} {err[i].mean():7.1f} mm {mgeo[i]:7.3f} {res['log_p'][i]:8.1f}")


def run(_):
    with out:
        out.clear_output(wait=True)
        chosen = [box.value for box in model_boxes]
        hidden = [NAMES22.index(x) for x in hidden_box.value]
        index, seed, n = pose_box.value, seed_box.value, n_box.value
        gt = test[index : index + 1]
        results = sample_all(chosen, gt, hidden, n, seed)
        hidden_text = ", ".join(hidden_box.value) or "none"
        print(f"test pose #{index}, seed {seed}, {n} samples per model, hidden: {hidden_text}")
        for res in results:
            note = ("   NOTE: trained with all joints visible: hidden joints are outside of what it has seen"
                    if hidden and not res["model"].masks else "")
            print(f"  model {res['label']}: {res['model'].description}{note}")
        print()
        print_comparison(gt, results, hidden)
        caption = f"{' vs '.join(chosen)} | hidden: {hidden_text} | test pose {index} | seed {seed} | {n} samples"
        fig = picture(gt, results, hidden, VIEWS[views_box.value], opacity_box.value, f"test pose #{index}, hidden: {hidden_text}", caption)
        plt.show()
        if joints_box.value:
            per_joint(gt, results, hidden)
            plt.show()
        if details_box.value:
            for res in results:
                print_samples(gt, res, hidden)
        slug = "_vs_".join(x.replace("paper flow ", "").split("/")[0].replace(" ", "_") for x in chosen)
        last.update(fig=fig, slug=f"{slug}_pose{index}_seed{seed}")
        say("")


def save(_):
    if "fig" not in last:
        say("sample first")
        return
    folder = expand(SAVE_DIR)
    folder.mkdir(parents=True, exist_ok=True)
    path, k = folder / f"{last['slug']}.png", 2
    while path.exists():
        path, k = folder / f"{last['slug']}_{k}.png", k + 1
    last["fig"].savefig(path, dpi=150, bbox_inches="tight")
    say(f"saved {path}")


preset_box.observe(set_preset, names="value")
random_button.on_click(guarded(random_pose))
refresh_button.on_click(guarded(refresh))
sample_button.on_click(guarded(run))
save_button.on_click(guarded(save))
set_preset()
display(widgets.VBox([
    widgets.HBox([*model_boxes, refresh_button]),
    widgets.HBox([widgets.VBox([preset_box, widgets.HBox([pose_box, random_button]), widgets.HBox([seed_box, n_box]),
                                views_box, widgets.HBox([opacity_box, joints_box, details_box]),
                                widgets.HBox([sample_button, save_button]), msg]),
                  hidden_box]),
    out]))

## 3. The occlusions of the paper on many poses

For `N_EVAL` random test poses, every model of `EVAL_MODELS` and every occlusion of `OCCLUSIONS`: `K` samples per pose, and

- **1 sample**: the metrics of the first sample;
- **mean of K**: the metrics of the mean pose of the K samples (the "N = 10" of the paper);
- **best of K**: the smallest error among the K samples (min-of-N, Tab. 10 and Fig. 4 of the paper).

MPJPE in mm over the 22 keypoints, MGEO in rad over the 19 joints. The paper's Tab. 3 reports "Ours (N = 10)" as MPJPE | MGEO: L 13.9 | 0.165, A+H 30.2 | 0.201, S+UA 12.7 | 0.162, on the authors' AMASS release (ours is SMPL-X G, see `docs/model_comparison_paper_protocol.md`). The conditional log-likelihood is left out: with the keypoints visible it is set by the smoothing of the model, not by the data, and is not comparable between models (`docs/rsgm_conditioning.md`, section 6).

In [ ]:
PAPER_TAB3 = {"L (left leg)": (13.9, 0.165), "A+H (left arm and hand)": (30.2, 0.201), "S+UA (right shoulder, upper arm)": (12.7, 0.162)}

g = torch.Generator().manual_seed(SEED + 1)
gt_eval = test[torch.randperm(len(test), generator=g)[:N_EVAL]]
kp_eval = keypoints(gt_eval)
results = {}
for name in (EVAL_MODELS or list(LOADERS)):
    model = get_model(name)
    for occlusion, joint_names in OCCLUSIONS.items():
        hidden = [AmassJoints[n].value for n in joint_names]
        s, _ = model.sample(kp_eval, hidden, K, SEED, with_log_prob=False)  # [N_EVAL, K, 19, 3, 3]
        kp_s = keypoints(s.reshape(-1, J, 3, 3)).reshape(N_EVAL, K, 22, 3)
        mpjpe = (kp_s - kp_eval[:, None]).norm(dim=-1).mean(-1) * 1000      # [N_EVAL, K]
        mgeo = roma.rotmat_geodesic_distance(s, gt_eval[:, None]).mean(-1)
        m = mean_rotation(s)
        results[name, occlusion] = dict(
            mpjpe_1=mpjpe[:, 0].mean(), mgeo_1=mgeo[:, 0].mean(),
            mpjpe_mean=((keypoints(m) - kp_eval).norm(dim=-1).mean(-1) * 1000).mean(), mgeo_mean=roma.rotmat_geodesic_distance(m, gt_eval).mean(-1).mean(),
            mpjpe_best=mpjpe.min(1).values.mean(), mgeo_best=mgeo.min(1).values.mean())

print(f"{N_EVAL} test poses, {K} samples each. MPJPE [mm] | MGEO [rad]\n")
for name in (EVAL_MODELS or list(LOADERS)):
    print(f"{name}  ({get_model(name).description})")
    print(f"  {'occlusion':34s} {'1 sample':>15s} {f'mean of {K}':>15s} {f'best of {K}':>15s} {'paper N=10':>15s}")
    for occlusion in OCCLUSIONS:
        r = results[name, occlusion]
        paper = PAPER_TAB3.get(occlusion)
        paper = f"{paper[0]:5.1f} | {paper[1]:.3f}" if paper else "-"
        print(f"  {occlusion:34s} {r['mpjpe_1']:5.1f} | {r['mgeo_1']:.3f} {r['mpjpe_mean']:7.1f} | {r['mgeo_mean']:.3f} "
              f"{r['mpjpe_best']:7.1f} | {r['mgeo_best']:.3f} {paper:>15s}")
    print()

# the models side by side: one table per metric, the lowest error of a row marked with *
evaluated = EVAL_MODELS or list(LOADERS)
for metric, label in (("mean", f"mean of {K}"), ("1", "1 sample"), ("best", f"best of {K}")):
    print(f"{label}: MPJPE [mm] | MGEO [rad]")
    print(f"  {'occlusion':34s}" + "".join(f"{f'model {k + 1}':>17s}" for k in range(len(evaluated))) + f"{'paper N=10':>17s}")
    for occlusion in OCCLUSIONS:
        values = [(float(results[name, occlusion][f"mpjpe_{metric}"]), float(results[name, occlusion][f"mgeo_{metric}"])) for name in evaluated]
        best = (int(np.argmin([v[0] for v in values])), int(np.argmin([v[1] for v in values])))
        cells = "".join(f" {v[0]:7.1f}{'*' if k == best[0] else ' '}| {v[1]:.3f}{'*' if k == best[1] else ' '}" for k, v in enumerate(values))
        paper = PAPER_TAB3.get(occlusion) if metric == "mean" else None
        print(f"  {occlusion:34s}{cells}{(f'{paper[0]:9.1f} | {paper[1]:.3f}' if paper else f'{chr(45):>17s}')}")
    print()
print("models: " + "; ".join(f"{k + 1}: {name}" for k, name in enumerate(evaluated)))